# Load factor analysis for chaining

Every hash table has one number that sets its speed and its memory at once: the load factor, keys per bucket. Libraries ship it as a constant, 0.75 in Java, and rarely say where it came from. This post derives one for a table that resolves collisions by chaining.

The birthday paradox opens it: collisions cannot be designed away, so a table has to price them instead. For chaining the price of an operation depends on the load factor and nothing else, and the memory a table spends is its reciprocal. Two rules turn those two prices into a number, one that bounds the time and one that weighs time against memory, and the post shows both because each exposes an assumption the other hides. A minimal chained hash map measures the claims along the way.

## Collisions are certain

Put $n$ keys into $m$ buckets with a hash that spreads them uniformly. The chance that no two share a bucket is

$$P(\text{no collision}) = \prod_{i=0}^{n-1}\left(1 - \frac{i}{m}\right) \approx \exp\!\left(-\sum_{i=0}^{n-1}\frac{i}{m}\right) = \exp\!\left(-\frac{n(n-1)}{2m}\right),$$

using $1 - x \approx e^{-x}$ for small $x$. It falls below one half once $n(n-1)/2m$ exceeds $\ln 2$, that is at

$$n \approx \sqrt{2 m \ln 2} \approx 1.18\sqrt{m}.$$

With $m = 365$ that is 23: the birthday paradox. With $m = 1021$ it is 38 keys, in a table that is 96% empty. Keeping collisions rare would take a bucket count of order $n^2$, so no table tries. A table accepts collisions and controls what they cost, and the control is the load factor $\alpha = n/m$.

## A minimal hash map

Chaining keeps a list per bucket. An insert hashes the key to a bucket, walks the chain comparing against every key already there, and appends if none matches. A lookup walks the same chain until it finds the key or runs out. The hash only has to spread keys evenly; the one below is a textbook universal hash, a random multiplier and offset modulo a large prime, then modulo the bucket count. It is not the subject here.

Two things in the code matter for the analysis. The table counts every key comparison it makes, because that is the unit the analysis prices. And it grows: when the load factor passes `max_load_factor`, the bucket count moves to the next prime in `PRIMES`, roughly doubling, and every key is rehashed. Growth can be switched off to hold the load factor still for a measurement.

In [1]:
import random

PRIMES = [3, 7, 13, 31, 61, 127, 251, 509, 1021, 2039, 4093, 8191]
MERSENNE_PRIME = (1 << 61) - 1


class HashMap:
    def __init__(self, bucket_count=PRIMES[0], max_load_factor=0.75, grow=True):
        self.bucket_count = bucket_count
        self.max_load_factor = max_load_factor
        self.grow = grow
        self.size = 0
        self.comparisons = 0
        self.buckets = [[] for _ in range(self.bucket_count)]
        self.multiplier = random.randrange(1, MERSENNE_PRIME)
        self.offset = random.randrange(0, MERSENNE_PRIME)

    @property
    def load_factor(self):
        return self.size / self.bucket_count

    def _bucket(self, key):
        return ((self.multiplier * key + self.offset) % MERSENNE_PRIME) % self.bucket_count

    def put(self, key, value):
        chain = self.buckets[self._bucket(key)]
        for i, (stored_key, _) in enumerate(chain):
            self.comparisons += 1
            if stored_key == key:
                chain[i] = (key, value)
                return
        chain.append((key, value))
        self.size += 1
        if self.grow and self.load_factor > self.max_load_factor:
            self._resize()

    def get(self, key):
        for stored_key, value in self.buckets[self._bucket(key)]:
            self.comparisons += 1
            if stored_key == key:
                return value
        raise KeyError(key)

    def _resize(self):
        next_prime = next((p for p in PRIMES if p > self.bucket_count), None)
        if next_prime is None:
            return
        self.bucket_count = next_prime
        pairs = [pair for chain in self.buckets for pair in chain]
        self.buckets = [[] for _ in range(self.bucket_count)]
        for stored_key, value in pairs:
            self.buckets[self._bucket(stored_key)].append((stored_key, value))

## The price of one operation

Hold the bucket count at $m$ and insert $n$ keys. Each key lands in a bucket independently with probability $1/m$, so the number of keys in a given bucket is $\text{Binomial}(n, 1/m)$, which for any real table is the Poisson distribution with mean $\alpha = n/m$. That one distribution prices everything.

A miss, a lookup for a key that is absent, walks its whole chain: $\alpha$ comparisons on average. An insert of a new key is a miss followed by an append, so it costs the same. A hit stops at the key it wants, on average halfway down a chain that also contains it:

$$E[\text{miss}] = \alpha, \qquad E[\text{hit}] \approx 1 + \frac{\alpha}{2}.$$

Neither depends on $n$ or $m$ on its own. Ten keys in fifteen buckets and ten million in fifteen million cost the same, and that is the whole reason a hash table is $O(1)$: it holds $\alpha$ constant whatever $n$ does.

The tail of the same distribution says how bad one bucket can get. At $\alpha = 0.75$ a chain of eight or more appears in about one bucket in a million; at $\alpha = 2$ in one bucket in a thousand. Java's HashMap turns a bucket into a tree at eight for this reason: at its load factor the case is rare enough to be an exception rather than the design.

The figure measures all of it on the map above. Growth is off and the bucket count is fixed at 1021. For each load factor from 0.05 to 2, in steps of 0.05, the table is filled, then probed with a thousand misses and a thousand hits, over a hundred trials. The lines are the formulas. The slider picks a load factor and shows what share of buckets hold 0, 1, 2, ... keys against the Poisson prediction, on a log scale so the tail is visible; the measured bars stop where a hundred thousand buckets run out of samples, the formula does not.

In [2]:
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

random.seed(0)
BUCKET_COUNT = PRIMES[8]
LOAD_FACTORS = [round(0.05 * step, 2) for step in range(1, 41)]
TRIALS = 100
PROBES = 1000
MAX_CHAIN = 10
KEY_SPACE = range(10_000_000)

miss_cost, hit_cost, occupancy = [], [], {}
for alpha in LOAD_FACTORS:
    key_count = round(alpha * BUCKET_COUNT)
    hit_probes = min(PROBES, key_count)
    miss_total = hit_total = 0
    counts = [0] * (MAX_CHAIN + 1)
    for _ in range(TRIALS):
        table = HashMap(bucket_count=BUCKET_COUNT, grow=False)
        keys = random.sample(KEY_SPACE, key_count + PROBES)
        present, absent = keys[:key_count], keys[key_count:]
        for key in present:
            table.put(key, None)
        table.comparisons = 0
        for key in absent:
            try:
                table.get(key)
            except KeyError:
                pass
        miss_total += table.comparisons
        table.comparisons = 0
        for key in random.sample(present, hit_probes):
            table.get(key)
        hit_total += table.comparisons
        for chain in table.buckets:
            counts[min(len(chain), MAX_CHAIN)] += 1
    miss_cost.append(miss_total / (TRIALS * PROBES))
    hit_cost.append(hit_total / (TRIALS * hit_probes))
    occupancy[alpha] = [c / (TRIALS * BUCKET_COUNT) for c in counts]


def poisson(alpha):
    pmf = [math.exp(-alpha) * alpha**k / math.factorial(k) for k in range(MAX_CHAIN)]
    return pmf + [1 - sum(pmf)]


def marker(alpha):
    return [dict(type="line", xref="x", yref="paper", x0=alpha, x1=alpha, y0=0, y1=1,
                 line=dict(color="crimson", dash="dot"))]


chain_lengths = list(range(MAX_CHAIN + 1))
start = 0.75
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.12,
                    subplot_titles=("comparisons per operation", "keys per bucket at the chosen load factor"))
fig.add_trace(go.Scatter(x=LOAD_FACTORS, y=LOAD_FACTORS, mode="lines", name="miss: α",
                         line=dict(color="#999999", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=LOAD_FACTORS, y=[1 + a / 2 for a in LOAD_FACTORS], mode="lines", name="hit: 1 + α/2",
                         line=dict(color="#999999", dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=LOAD_FACTORS, y=miss_cost, mode="markers", name="miss, measured",
                         marker=dict(color="crimson")), row=1, col=1)
fig.add_trace(go.Scatter(x=LOAD_FACTORS, y=hit_cost, mode="markers", name="hit, measured",
                         marker=dict(color="#1f77b4")), row=1, col=1)
fig.add_trace(go.Bar(x=chain_lengths, y=occupancy[start], name="share of buckets, measured",
                     marker=dict(color="#1f77b4")), row=1, col=2)
fig.add_trace(go.Scatter(x=chain_lengths, y=poisson(start), mode="lines+markers", name="Poisson(α)",
                         line=dict(color="crimson")), row=1, col=2)

steps = [dict(method="update", label=str(alpha) if (alpha * 4).is_integer() else "",
              args=[{"y": [occupancy[alpha], poisson(alpha)]}, {"shapes": marker(alpha)}, [4, 5]])
         for alpha in LOAD_FACTORS]
fig.update_layout(
    sliders=[dict(active=LOAD_FACTORS.index(start), currentvalue={"prefix": "load factor α = "}, steps=steps)],
    shapes=marker(start),
    legend=dict(orientation="h", y=-0.45),
    margin=dict(t=50),
)
fig.update_xaxes(title_text="load factor α", row=1, col=1)
fig.update_yaxes(title_text="key comparisons", row=1, col=1)
fig.update_xaxes(title_text="keys in a bucket", tickvals=chain_lengths,
                 ticktext=[str(k) for k in range(MAX_CHAIN)] + [f"{MAX_CHAIN}+"], row=1, col=2)
fig.update_yaxes(title_text="share of buckets", type="log", range=[-6, 0], row=1, col=2)
fig.show()

## The price of a small load factor

The bucket array costs $m$ slots for $n$ keys, that is $1/\alpha$ slots per key on top of the nodes that hold the keys. Halving the load factor doubles that overhead. Past $\alpha = 1$ the array is already smaller than the data it indexes, so pushing higher saves less than a slot per key while every operation gets linearly slower; that is why the figure keeps climbing to the right and why no library goes there.

A growing table never sits at one load factor. Doubling when $\alpha$ passes the cap $\alpha_{\max}$ drops it to $\alpha_{\max}/2$, so the table lives between the two and averages about $0.56$ for a cap of $0.75$. The rehashes are cheap in total: each doubling rehashes every key once, and the sum over all doublings is under two rehashes per key ever inserted.

The formulas above, evaluated at the caps a designer would consider:

In [3]:
import math
from IPython.display import HTML


def chain_at_least(length, alpha):
    return 1 - sum(math.exp(-alpha) * alpha**k / math.factorial(k) for k in range(length))


rows = "".join(
    f"<tr><td>{alpha}</td><td>{alpha:.2f}</td><td>{1 + alpha / 2:.3f}</td>"
    f"<td>{chain_at_least(8, alpha):.1e}</td><td>{1 / alpha:.2f}</td></tr>"
    for alpha in (0.5, 0.75, 1.0, 1.5, 2.0)
)
HTML("<table><thead><tr><th>cap α</th><th>miss</th><th>hit</th><th>P(chain ≥ 8)</th><th>slots per key</th></tr></thead>"
     f"<tbody>{rows}</tbody></table>")

cap α,miss,hit,P(chain ≥ 8),slots per key
0.5,0.50,1.250,6.2e-08,2.00
0.75,0.75,1.375,1.3e-06,1.33
1.0,1.00,1.500,1.0e-05,1.00
1.5,1.50,1.750,1.7e-04,0.67
2.0,2.00,2.000,1.1e-03,0.50


## Choosing the load factor

Two rules give a number, and they answer different questions.

**Bound the time, minimize the memory.** Treat speed as a promise and memory as the bill. Fix what an operation may cost and how rare a long chain must be, then take the largest $\alpha$ that keeps both, because every step higher saves memory for free until a bound breaks. With a miss capped at one comparison and a chain of eight rarer than one in a million, the first bound allows $\alpha \le 1$ and the second $\alpha \le 0.72$. The tail binds first, at $0.72$; rounded, that is the $0.75$ libraries ship, and its bill is $1.33$ slots per key at the cap and $2.67$ right after a doubling.

**Price both, minimize the sum.** Treat neither as a promise. Let a comparison cost $t$ and a slot cost $s$, in whatever unit you bill in, and minimize

$$C(\alpha) = t\,\alpha + \frac{s}{\alpha}, \qquad \alpha^\ast = \sqrt{\frac{s}{t}}.$$

If a slot and a comparison cost the same, $\alpha^\ast = 1$. If memory is four times cheaper than time, $\alpha^\ast = 0.5$. There is no constant here at all, only a ratio of prices.

Both rules are shown because each exposes what the other hides. The first produces a familiar number, but only after two thresholds were chosen by hand, and moving either moves the answer. The second has no thresholds, but it needs a price for memory relative to time, which is the thing nobody knows precisely and the reason the first rule exists. A shipped constant is one rule's output for one set of inputs. Change the inputs, say a lookup path where every comparison is a cache miss, and the same analysis gives a different constant.

## Conclusion

Collisions are certain by the time a table is a few percent full, so a table's job is not to avoid them but to price them. For chaining the price is one distribution, Poisson with mean $\alpha$: a miss costs $\alpha$ comparisons, a hit $1 + \alpha/2$, a chain of eight is a tail probability, and the memory overhead is $1/\alpha$ slots per key. Every one of those was measured on a forty-line map and matched.

The load factor is therefore not a tuning constant but the output of a decision. Bound the time and the largest passing $\alpha$ is about $0.72$, which is where $0.75$ comes from. Price memory against time and the answer is $\sqrt{s/t}$, a ratio rather than a constant. Pick the rule that matches what you are promising, and the number follows.

**[In-browser notebook →](/lite/notebooks/index.html?path=2026-09-24-load-factor-analysis-for-chaining.ipynb)**